## DSAN 6000 Homework 3A: Allocating Tasks to Parallel Workers with `joblib`

## Overview

You made it to the first DSAN 6000 homework introducing a new coding concept! The goal of this part is for you to gain hands-on experience using **`joblib`** to quickly parallelize an **embarrassingly-parallel** task.

In this case, the problem is one that relates back to Week 1 and Week 2 content on the **OnLine Transaction Processing (OLTP)** mode of data collection and processing: the setup is that a designer spoon brand has set up a new website, where users from across the globe can purchase their latest avant garde silverware using their credit cards. Information from these credit card transactions then **streams into their OLTP database** like you saw in the Week 1 demo!

...However, you've been hired by this designer spoon brand (given your reputation as a DSAN graduate expert) because the company has been experiencing an upsurge in the use of **fraudulent (fake) credit card numbers** being used during checkout! How do we detect fake credit card numbers? In the real world, credit card companies often employ something like the [Luhn Algorithm](https://en.wikipedia.org/wiki/Luhn_algorithm) to choose credit card numbers for customers, whereby only *some* of the 16 digits in the card number are randomly generated: the remaining digits are *computed* as "check" digits, via some mathematical operation applied to the digits that were randomly generated.

For this assignment, we have implemented a simplified version of the Luhn Algorithm scheme: the first 15 digits are randomly-generated (with each digit uniformly sampled from $\{0, 1, \ldots, 9\}$), and then the 16th digit is the **sum of these first 15 digits, mod 10**. Mathematically, if $d_1$ represents the first randomly-generated digit, $d_2$ the second randomly-generated digit, and so on:

$$
d_{16} = \left(\sum_{i=1}^{15}d_i\right) \text{mod }{10}
$$

Using this scheme, then, you can check for valid credit card numbers (in the context of this assignment... in the real world you would need to do more work!) by just verifying that the sum of the first 15 digits (modulo 10) is equal to the 16th digit.

The *computational* problem is that, checking for fraudulent card numbers **in serial** significantly slows down the site's transaction-processing throughput... which is why you were hired! So, your task will be to use **parallel processing** to see how/whether you can "push" this OLTP system to process a larger volume of transactions per unit of time.

## Part 1: Loading and Preparing Data

### Part 1.1: Use `boto3` to Download the `.parquet` File From S3

Since you already figured out how to use `boto3` to connect to and download files from an S3 bucket in HW2, here we have provided code for you importing `boto3` and setting up the `s3` client object. Your task is to use this client to **download** the file at the following URI:

```
s3://dsan6000-data/transactions_10m.parquet
```

To your EC2 instance, saving it to have the same filename in a `data` subfolder (within the `dsan6000-hw03-parallel-processing` folder). We have already included a `.gitignore` file in the template repo, to ensure that this `.parquet` file doesn't get pushed to GitHub, since it is larger than the allowed invidual file size on GitHub!

In [1]:
#| label: q1.1-init
import boto3
s3 = boto3.client('s3')

In [2]:
#| label: q1.1-response
# Your code here: Download transactions_10m.parquet to the data subfolder
s3.download_file('dsan6000-data', 'cc_transactions_10m.parquet', 'data/cc_transactions_10m.parquet')

### Part 1.2: Loading the Data Into Pandas

In [1]:
#| label: q1.2-init
import pandas as pd
import numpy as np

In [2]:
#| label: q1.2-response
oltp_df = pd.read_parquet("data/cc_transactions_10m.parquet")
oltp_df

,timestamp,customer_id,product_id,amount,cc_num
0,2026-08-20 09:11:41.546837,13927,59,70.30 PLN,3684979005201906
1,2026-08-20 09:11:48.636365,96584,45,47.90 JPY,8692661761705576
2,2026-08-20 09:11:48.713547,85012,70,21.67 ZAR,8079389766740026
3,2026-08-20 09:11:48.925565,96518,36,81.23 JPY,1196239118773592
4,2026-08-20 09:11:55.104031,29571,29,76.49 DKK,0218319760284966
...,...,...,...,...,...
9999995,2026-09-04 09:09:31.340697,70218,3,29.09 USD,1270634138434242
9999996,2026-09-04 09:09:31.411286,62114,65,66.59 MXN,6692724334565888
9999997,2026-09-04 09:09:31.554090,97054,29,32.54 JPY,5206767646252604
9999998,2026-09-04 09:09:31.560754,55995,67,35.19 KRW,5772540155003543


In [3]:
oltp_sub_df = oltp_df.iloc[:1_000_000].copy()

In [4]:
oltp_sub_df

,timestamp,customer_id,product_id,amount,cc_num
0,2026-08-20 09:11:41.546837,13927,59,70.30 PLN,3684979005201906
1,2026-08-20 09:11:48.636365,96584,45,47.90 JPY,8692661761705576
2,2026-08-20 09:11:48.713547,85012,70,21.67 ZAR,8079389766740026
3,2026-08-20 09:11:48.925565,96518,36,81.23 JPY,1196239118773592
4,2026-08-20 09:11:55.104031,29571,29,76.49 DKK,0218319760284966
...,...,...,...,...,...
999995,2026-08-21 21:14:10.136907,96678,84,36.20 MYR,6758553537840938
999996,2026-08-21 21:14:10.137788,41287,39,54.34 ZAR,4888412895951923
999997,2026-08-21 21:14:10.169447,4614,32,44.82 IDR,5176237239874598
999998,2026-08-21 21:14:10.170679,94782,55,63.83 JPY,2941022485827936


## Part 2: Verifying Credit Card Numbers in Serial

Here, the goal is explicitly *not* to do anything fancy: just use a standard for loop to process each of the 10 million transactions you just downloaded. To make the comparison with parallel processing as fair as possible, however, you should **extract just the `cc_num` column** from the full `DataFrame`, using the `to_list()` function available on Pandas `Series` objects to store these extracted values in a list named `cc_nums`. Then

In [5]:
import time
disp_time = lambda start, end: print('{:.4f} s'.format(end - start))


In [39]:
import hashlib
import math

def verify_cc_num(cc_num):
  cc_ints = np.array([int(d) for d in cc_num])
  # print(cc_ints)
  cc_sum = np.prod([np.pow(d+1, d+1) for d in cc_ints[:15]])
  # print(cc_sum)
  cc_mod = np.mod(cc_sum, 10)
  return bool(cc_mod == cc_ints[-1])

print(verify_cc_num('3684979005201906'))
print(verify_cc_num('8692661761705576'))

False
False


In [40]:
cc_nums = oltp_sub_df['cc_num'].to_list()
len(cc_nums)

1000000

In [41]:
from tqdm import tqdm

In [42]:
serial_start = time.time()
valid_ccs = [verify_cc_num(num) for num in tqdm(cc_nums)]
serial_end = time.time()
disp_time(serial_start, serial_end)

100%|██████████| 1000000/1000000 [00:33<00:00, 30093.77it/s]

33.2388 s


In [43]:
len(valid_ccs)

1000000

In [44]:
np.sum(valid_ccs)

np.int64(100175)

## Part 3: Converting Currencies in Parallel

In [45]:
import numpy as np
import joblib
joblib.cpu_count()

2

In [52]:
parallel_runner = joblib.Parallel(n_jobs=4, batch_size='auto', verbose=True)
par_start = time.time()
valid_ccs_parallel = parallel_runner(
  joblib.delayed(verify_cc_num)(num) for num in cc_nums
)
par_end = time.time()
disp_time(par_start, par_end)

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done 312 tasks      | elapsed:    0.3s
[Parallel(n_jobs=4)]: Done 229368 tasks      | elapsed:    8.7s
[Parallel(n_jobs=4)]: Done 741368 tasks      | elapsed:   27.1s


36.1830 s


[Parallel(n_jobs=4)]: Done 1000000 out of 1000000 | elapsed:   36.2s finished


You made it to the end of HW3A!

Before moving to HW3B, take note of the difference in total time required between the **serial** and **parallel** approaches, and answer the following questions:

batch=50
```
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done 2304 tasks      | elapsed:    0.4s
[Parallel(n_jobs=-1)]: Done 9804 tasks      | elapsed:    0.7s
[Parallel(n_jobs=-1)]: Done 22304 tasks      | elapsed:    1.9s
[Parallel(n_jobs=-1)]: Done 39804 tasks      | elapsed:    2.8s
[Parallel(n_jobs=-1)]: Done 62304 tasks      | elapsed:    3.4s
[Parallel(n_jobs=-1)]: Done 89804 tasks      | elapsed:    4.6s
[Parallel(n_jobs=-1)]: Done 122304 tasks      | elapsed:    5.5s
[Parallel(n_jobs=-1)]: Done 159804 tasks      | elapsed:    6.5s
[Parallel(n_jobs=-1)]: Done 202304 tasks      | elapsed:    7.7s
[Parallel(n_jobs=-1)]: Done 249804 tasks      | elapsed:    8.9s
[Parallel(n_jobs=-1)]: Done 302304 tasks      | elapsed:   10.4s
[Parallel(n_jobs=-1)]: Done 359804 tasks      | elapsed:   11.9s
[Parallel(n_jobs=-1)]: Done 422304 tasks      | elapsed:   13.6s
[Parallel(n_jobs=-1)]: Done 489804 tasks      | elapsed:   15.4s
[Parallel(n_jobs=-1)]: Done 562304 tasks      | elapsed:   17.4s
[Parallel(n_jobs=-1)]: Done 639804 tasks      | elapsed:   19.4s
[Parallel(n_jobs=-1)]: Done 722304 tasks      | elapsed:   21.6s
[Parallel(n_jobs=-1)]: Done 809804 tasks      | elapsed:   24.0s
[Parallel(n_jobs=-1)]: Done 902304 tasks      | elapsed:   26.4s
[Parallel(n_jobs=-1)]: Done 999804 tasks      | elapsed:   28.8s
[Parallel(n_jobs=-1)]: Done 1102304 tasks      | elapsed:   31.5s
[Parallel(n_jobs=-1)]: Done 1209804 tasks      | elapsed:   34.3s
[Parallel(n_jobs=-1)]: Done 1322304 tasks      | elapsed:   37.3s
[Parallel(n_jobs=-1)]: Done 1439804 tasks      | elapsed:   40.5s
[Parallel(n_jobs=-1)]: Done 1562304 tasks      | elapsed:   43.7s
```

batch=200
```
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done 9204 tasks      | elapsed:    0.5s
[Parallel(n_jobs=-1)]: Done 39204 tasks      | elapsed:    1.3s
[Parallel(n_jobs=-1)]: Done 89204 tasks      | elapsed:    2.2s
[Parallel(n_jobs=-1)]: Done 159204 tasks      | elapsed:    3.3s
[Parallel(n_jobs=-1)]: Done 249204 tasks      | elapsed:    4.9s
[Parallel(n_jobs=-1)]: Done 359204 tasks      | elapsed:    6.7s
[Parallel(n_jobs=-1)]: Done 489204 tasks      | elapsed:    8.9s
[Parallel(n_jobs=-1)]: Done 639204 tasks      | elapsed:   11.4s
[Parallel(n_jobs=-1)]: Done 809204 tasks      | elapsed:   14.5s
[Parallel(n_jobs=-1)]: Done 999204 tasks      | elapsed:   17.6s
[Parallel(n_jobs=-1)]: Done 1209204 tasks      | elapsed:   21.2s
[Parallel(n_jobs=-1)]: Done 1439204 tasks      | elapsed:   25.1s
[Parallel(n_jobs=-1)]: Done 1689204 tasks      | elapsed:   29.3s
[Parallel(n_jobs=-1)]: Done 1959204 tasks      | elapsed:   33.9s
[Parallel(n_jobs=-1)]: Done 2249204 tasks      | elapsed:   38.8s
[Parallel(n_jobs=-1)]: Done 2559204 tasks      | elapsed:   44.5s
[Parallel(n_jobs=-1)]: Done 2889204 tasks      | elapsed:   49.9s
[Parallel(n_jobs=-1)]: Done 3239204 tasks      | elapsed:   55.6s
[Parallel(n_jobs=-1)]: Done 3609204 tasks      | elapsed:  1.0min
[Parallel(n_jobs=-1)]: Done 3999204 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done 4409204 tasks      | elapsed:  1.3min
```

batch=300
```
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done 13804 tasks      | elapsed:    0.6s
[Parallel(n_jobs=-1)]: Done 58804 tasks      | elapsed:    2.3s
[Parallel(n_jobs=-1)]: Done 133804 tasks      | elapsed:    3.4s
[Parallel(n_jobs=-1)]: Done 238804 tasks      | elapsed:    5.0s
[Parallel(n_jobs=-1)]: Done 373804 tasks      | elapsed:    7.1s
[Parallel(n_jobs=-1)]: Done 538804 tasks      | elapsed:    9.6s
[Parallel(n_jobs=-1)]: Done 733804 tasks      | elapsed:   12.5s
[Parallel(n_jobs=-1)]: Done 958804 tasks      | elapsed:   15.9s
[Parallel(n_jobs=-1)]: Done 1213804 tasks      | elapsed:   19.8s
[Parallel(n_jobs=-1)]: Done 1498804 tasks      | elapsed:   24.3s
[Parallel(n_jobs=-1)]: Done 1813804 tasks      | elapsed:   29.0s
[Parallel(n_jobs=-1)]: Done 2158804 tasks      | elapsed:   34.2s
[Parallel(n_jobs=-1)]: Done 2533804 tasks      | elapsed:   40.0s
[Parallel(n_jobs=-1)]: Done 2938804 tasks      | elapsed:   46.3s
[Parallel(n_jobs=-1)]: Done 3373804 tasks      | elapsed:   53.0s
```

batch=800
```
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done 36804 tasks      | elapsed:    1.8s
[Parallel(n_jobs=-1)]: Done 156804 tasks      | elapsed:    4.1s
[Parallel(n_jobs=-1)]: Done 356804 tasks      | elapsed:    7.3s
[Parallel(n_jobs=-1)]: Done 636804 tasks      | elapsed:   12.0s
[Parallel(n_jobs=-1)]: Done 996804 tasks      | elapsed:   17.9s
[Parallel(n_jobs=-1)]: Done 1436804 tasks      | elapsed:   25.3s
```

batch=4000
```
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done 184004 tasks      | elapsed:    4.5s
[Parallel(n_jobs=-1)]: Done 784004 tasks      | elapsed:   16.6s
[Parallel(n_jobs=-1)]: Done 1784004 tasks      | elapsed:   37.5s
[Parallel(n_jobs=-1)]: Done 3184004 tasks      | elapsed:  1.1min
```

batch=8000
```
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done 368004 tasks      | elapsed:    7.7s
[Parallel(n_jobs=-1)]: Done 1568004 tasks      | elapsed:   32.5s
```

batch=20000
```
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done 920004 tasks      | elapsed:   18.1s
[Parallel(n_jobs=-1)]: Done 3920004 tasks      | elapsed:  1.3min
```